# 08 — Boosting Models

## Objectif

L'objectif de ce notebook est d'évaluer plusieurs modèles de boosting sur le challenge QRT afin de déterminer si une stratégie d'apprentissage séquentielle permet de mieux exploiter les rendements historiques que les modèles étudiés précédemment.

Contrairement au bagging, qui cherche principalement à réduire la variance en agrégeant plusieurs modèles indépendants, le boosting construit les modèles de manière séquentielle. Chaque nouvel arbre tente d'améliorer les prédictions produites par les arbres précédents.

Tous les modèles sont évalués avec le même protocole de validation temporelle afin de garantir une comparaison équitable avec les expériences précédentes.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import load_X_train, load_y_train
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.evaluation import evaluate_model_on_folds, compare_model_results
from src.boosting_models import build_adaboost_pipeline, build_gradboosting_pipeline
from src.target import create_class_column
from src.features import (create_momentum_features, 
                          create_volatility_features, 
                          create_momentum_volatility_ratio_features,
                          create_volume_features)

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train, _ = create_class_column(X_train, y_train)

In [5]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [6]:
df_train = create_momentum_features(df_train)
df_train = create_volatility_features(df_train)
df_train = create_momentum_volatility_ratio_features(df_train)
df_train = create_volume_features(df_train)

In [7]:
df_train.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'momentum_3', 'momentum_5', 'momentum_10',
       'momentum_20', 'delta_momentum_3_20', 'volatility_3', 'volatility_20',
       'delta_volatility_3_20', 'momentum_volatility_ratio_20', 'volume_3',
       'volume_20', 'delta_volume_3_20'],
      dtype='str')

In [8]:
drop_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'class']

In [9]:
ret_features = [
    'RET_20', 'RET_19', 'RET_18', 'RET_17',
    'RET_16', 'RET_15', 'RET_14', 'RET_13', 
    'RET_12', 'RET_11', 'RET_10','RET_9', 
    'RET_8', 'RET_7', 'RET_6', 'RET_5', 
    'RET_4', 'RET_3', 'RET_2','RET_1'
]

In [10]:
features = df_train.drop(
    columns = drop_cols
).columns

## AdaBoost

### Hypothèse

AdaBoost entraîne une succession de petits arbres de décision (Decision Stumps). Après chaque itération, les observations mal classées reçoivent un poids plus important afin que le modèle suivant concentre davantage son apprentissage sur ces exemples difficiles.

L'hypothèse testée est que cette stratégie peut permettre d'améliorer les performances par rapport à un arbre unique ou à une Random Forest en exploitant progressivement les erreurs commises lors des itérations précédentes.

In [11]:
adaboost = build_adaboost_pipeline()

In [ ]:
adaboost_results = evaluate_model_on_folds(df_train,folds,adaboost,features)

In [ ]:
adaboost_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.518578,0.739541,0.520816,24114,12505
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.532464,0.729316,0.549135,24396,12990
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.519100,0.740114,0.521695,24764,12855
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.522837,0.741141,0.524957,25660,13416


### Interprétation

Les performances obtenues avec AdaBoost restent proches de celles des modèles précédents.

L'Accuracy et le ROC-AUC montrent une légère amélioration par rapport à l'arbre de décision, mais le Log-Loss est nettement plus élevé. Cela indique que les probabilités produites par AdaBoost sont moins bien calibrées et que le modèle tend à être trop confiant lorsqu'il se trompe.

Cette observation est cohérente avec les limites connues d'AdaBoost : le mécanisme de repondération des observations améliore souvent la classification mais peut conduire à des probabilités moins fiables.

Compte tenu de ces résultats, AdaBoost n'est pas retenu comme modèle final pour ce projet.

## Gradient Boosting

### Hypothèse

Contrairement à AdaBoost, Gradient Boosting ne se contente pas de repondérer les observations mal classées.

À chaque itération, un nouvel arbre apprend à corriger les erreurs résiduelles du modèle précédent en suivant le gradient de la fonction de perte. Cette approche permet d'effectuer des corrections progressives et de construire un modèle additif capable de capturer des relations non linéaires plus complexes.

L'objectif est de déterminer si cette stratégie améliore les performances sur les rendements historiques par rapport aux modèles précédemment étudiés.

In [ ]:
gradboost = build_gradboosting_pipeline()

In [ ]:
gradboost_results = evaluate_model_on_folds(df_train,folds,gradboost,features)

In [ ]:
gradboost_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.520445,0.692142,0.521002,24114,12550
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.535457,0.690796,0.547587,24396,13063
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.524188,0.692083,0.522158,24764,12981
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.523422,0.691702,0.524611,25660,13431


In [ ]:
results_str_cols = [
    'fold', 'train_start', 
    'train_end', 'valid_start', 
    'valid_end'
]

In [ ]:
gradboost_results.drop(
    columns=results_str_cols
).mean(axis=0)

train_positive_rate          0.505661
valid_positive_rate          0.517429
accuracy                     0.525878
log_loss                     0.691681
roc_auc                      0.528839
n_valid_predictions      24733.500000
n_correct_predictions    13006.250000
dtype: float64

### Interprétation

Le Gradient Boosting obtient les meilleures performances observées jusqu'à présent parmi les modèles implémentés.

Une légère amélioration est observée sur l'Accuracy tout en conservant un Log-Loss comparable à celui de la Random Forest. Ces résultats suggèrent que l'apprentissage séquentiel des corrections est plus adapté au problème étudié que la simple repondération des observations utilisée par AdaBoost.

Même si le gain reste modeste, cette amélioration est cohérente sur plusieurs folds de validation temporelle.

# Conclusion

Cette étape a permis d'étudier deux grandes familles de modèles de boosting : AdaBoost et Gradient Boosting.

Les principaux enseignements sont les suivants :

- AdaBoost améliore légèrement la classification mais produit des probabilités moins bien calibrées.
- Gradient Boosting obtient les meilleures performances observées jusqu'à présent sur les rendements historiques.
- Un premier ajustement des hyperparamètres n'apporte pas de gain significatif supplémentaire.
- La première soumission officielle basée sur Gradient Boosting dépasse le benchmark public du challenge, confirmant que cette famille de modèles exploite mieux le signal contenu dans les rendements historiques.

La prochaine étape consistera à étudier des implémentations plus modernes du boosting (XGBoost, LightGBM et CatBoost), qui sont spécifiquement conçues pour améliorer les performances et réduire le coût de calcul sur les grands jeux de données.